# Day 1 — MLOps Foundations (Practice)

Companion notebook for [day1_materials.md](day1_materials.md).

## Contents
- **Lab 0** — Environment sanity check
- **Lab 1** — Reproducibility: seeds, environment, project structure ([theory](day1_materials.md#module-2--reproducibility--environments-1045--1215))
- **Lab 2** — Experiment tracking with MLflow ([theory](day1_materials.md#module-3--experiment-tracking-with-mlflow-1315--1445))
- **Lab 3** — A tiny end-to-end pipeline + data validation ([theory](day1_materials.md#module-4--data-versioning-validation-pipelines-1500--1630))

> Run cells top-to-bottom. Discussion prompts are flagged with **DISCUSS**.

## Lab 0 — Environment sanity check

Make sure required libraries are installed (see `../requirements.txt`).

In [ ]:
import sys, platform
import numpy, pandas, sklearn, mlflow

print('Python    :', sys.version.split()[0], 'on', platform.system())
print('numpy     :', numpy.__version__)
print('pandas    :', pandas.__version__)
print('sklearn   :', sklearn.__version__)
print('mlflow    :', mlflow.__version__)

## Lab 1 — Reproducibility

Goal: run the same training twice and get **identical** numbers.

Reference: [Day 1, Module 2](day1_materials.md#module-2--reproducibility--environments-1045--1215)

In [ ]:
import os, random
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def set_seed(seed: int = 42) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

def train_once(seed: int = 42) -> float:
    set_seed(seed)
    X, y = load_breast_cancer(return_X_y=True)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )
    model = LogisticRegression(max_iter=5000, random_state=seed)
    model.fit(X_tr, y_tr)
    return roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])

auc1 = train_once()
auc2 = train_once()
print(f'run 1 AUC = {auc1:.6f}')
print(f'run 2 AUC = {auc2:.6f}')
assert auc1 == auc2, 'Not reproducible!'
print('OK — runs are bit-for-bit identical.')

**Exercise 1.1** — Remove the `random_state` arguments and re-run. What changes? Why?

**Exercise 1.2** — Print the current git commit using `subprocess`. Why might you log this with every experiment?

> **DISCUSS:** what other sources of non-determinism would appear if this were a deep learning model on GPU?

## Lab 2 — Experiment tracking with MLflow

We will train several models with different hyperparameters and compare them in the MLflow UI.

Reference: [Day 1, Module 3](day1_materials.md#module-3--experiment-tracking-with-mlflow-1315--1445)

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier

mlflow.set_tracking_uri('file:./mlruns')   # local file-based tracking
mlflow.set_experiment('day1-breast-cancer')

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

param_grid = [
    {'n_estimators': 50,  'max_depth': 3},
    {'n_estimators': 100, 'max_depth': 5},
    {'n_estimators': 200, 'max_depth': None},
]

for params in param_grid:
    with mlflow.start_run(run_name=f"rf_n{params['n_estimators']}_d{params['max_depth']}"):
        mlflow.log_params(params)
        model = RandomForestClassifier(random_state=42, **params)
        model.fit(X_tr, y_tr)
        auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
        mlflow.log_metric('auc', auc)
        mlflow.sklearn.log_model(model, name='model')
        print(f'logged {params} -> auc={auc:.4f}')

Launch the MLflow UI from a terminal in this folder:

```bash
mlflow ui --backend-store-uri file:./mlruns
```

Open http://127.0.0.1:5000 and:
1. Sort runs by `auc`.
2. Open the best run and inspect parameters and the logged model artifact.
3. Compare two runs side-by-side.

**Exercise 2.1** — Add a tag for your name (`mlflow.set_tag('owner', 'alice')`).

**Exercise 2.2** — Programmatically find the best run:

In [ ]:
from mlflow.tracking import MlflowClient
client = MlflowClient()
exp = client.get_experiment_by_name('day1-breast-cancer')
runs = client.search_runs(exp.experiment_id, order_by=['metrics.auc DESC'], max_results=5)
for r in runs:
    print(r.data.metrics.get('auc'), r.data.params)

**Exercise 2.3** — Register the best run's model in the MLflow Model Registry under the name `breast-cancer-classifier` and transition it to the `Staging` stage.

> **DISCUSS:** what would change if multiple data scientists were logging to the same tracking server?

## Lab 3 — A tiny end-to-end pipeline with data validation

We split the workflow into clearly-separated functions: **load → validate → features → train → evaluate**. This is the embryo of a real pipeline.

Reference: [Day 1, Module 4](day1_materials.md#module-4--data-versioning-validation-pipelines-1500--1630)

In [ ]:
from dataclasses import dataclass
import pandas as pd

@dataclass
class DataValidationError(Exception):
    message: str

def load() -> pd.DataFrame:
    data = load_breast_cancer(as_frame=True)
    df = data.frame
    return df

EXPECTED_COLS = set(load_breast_cancer(as_frame=True).frame.columns)

def validate(df: pd.DataFrame) -> pd.DataFrame:
    missing = EXPECTED_COLS - set(df.columns)
    if missing:
        raise DataValidationError(f'missing columns: {missing}')
    if df.isna().any().any():
        raise DataValidationError('found null values')
    if not df['target'].isin([0, 1]).all():
        raise DataValidationError('target must be binary 0/1')
    if len(df) < 100:
        raise DataValidationError(f'too few rows: {len(df)}')
    return df

def split(df: pd.DataFrame):
    y = df['target'].to_numpy()
    X = df.drop(columns=['target']).to_numpy()
    return train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def train(X_tr, y_tr) -> RandomForestClassifier:
    model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    model.fit(X_tr, y_tr)
    return model

def evaluate(model, X_te, y_te) -> dict:
    return {'auc': float(roc_auc_score(y_te, model.predict_proba(X_te)[:, 1]))}

def run_pipeline() -> dict:
    df = load()
    df = validate(df)
    X_tr, X_te, y_tr, y_te = split(df)
    model = train(X_tr, y_tr)
    metrics = evaluate(model, X_te, y_te)
    return {'model': model, 'metrics': metrics}

result = run_pipeline()
print(result['metrics'])

**Exercise 3.1** — Break it on purpose. Drop a required column or inject a `NaN` and confirm `validate()` raises before training.

**Exercise 3.2** — Add a check that the **distribution** of one feature is roughly within the training reference (mean ± 3·std). This is a baseline for drift detection on Day 2.

**Exercise 3.3** — Wrap `run_pipeline()` in MLflow logging so each pipeline run is tracked end-to-end (params, metrics, model).

## Day 1 wrap-up

You can now:
- Make a training script reproducible.
- Track and compare experiments in MLflow.
- Validate data at the boundary of your pipeline.
- Express training as a small DAG of pure functions.

**Tomorrow:** packaging, serving via REST, CI/CD for ML, and monitoring for drift. See [day2_materials.md](../day2/day2_materials.md).